# Ngày 12 · SVM: lề, hinge loss và kernel

- Viết hàm mục tiêu SVM lề mềm ½‖w‖² + CΣmax(0, 1 − y(w·x + b))
- Tính subgradient và huấn luyện bằng subgradient descent
- Hiểu vai trò của C và vector hỗ trợ

**Đọc:**
- ISLR, Chương 9 Support Vector Machines, §9.1–9.3
- Stanford CS229, Chương 6 (lề, bài toán đối ngẫu, kernel)
- Vũ Hữu Tiệp, Machine Learning cơ bản, Chương 26–28 (SVM, lề mềm, hạt nhân)

In [ ]:
import json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

DATA = next((p for p in (Path('data'), Path('../data')) if p.is_dir()), Path('data'))
# Breast Cancer Wisconsin (Diagnostic), UCI, có sẵn trong scikit-learn. 569 mẫu, 30 đặc trưng.
_bc = load_breast_cancer()
X_bc = _bc.data
y_bc = 1 - _bc.target   # 1 = ác tính (lớp cần phát hiện), 0 = lành tính
feature_names = list(_bc.feature_names)

def _check(name, fn):
    try:
        ok, detail = fn()
    except NotImplementedError:
        ok, detail = False, 'Chưa cài đặt (NotImplementedError)'
    except Exception as error:
        ok, detail = False, f'{type(error).__name__}: {error}'
    return {'name': name, 'ok': bool(ok), 'detail': str(detail)}

def _report(ex_id, *checks):
    print('__PY30__' + json.dumps({'id': ex_id, 'checks': list(checks)}, ensure_ascii=False))

# Một phép chia dùng chung cho cả tuần: 70% train, 30% validation, phân tầng theo nhãn.
idx_tr, idx_va = train_test_split(np.arange(len(y_bc)), test_size=0.3, random_state=0, stratify=y_bc)
_scaler = StandardScaler().fit(X_bc[idx_tr])
X_tr, X_va = _scaler.transform(X_bc[idx_tr]), _scaler.transform(X_bc[idx_va])
y_tr, y_va = y_bc[idx_tr], y_bc[idx_va]
from sklearn.svm import SVC
_X12 = X_bc[:, [0, 1]]; _X12 = (_X12 - _X12.mean(0)) / _X12.std(0)
_y12 = np.where(y_bc == 1, 1., -1.)
_svc = SVC(kernel='linear', C=1.0).fit(_X12, _y12)
def _obj(w, b, X, y, C):
    return 0.5 * w @ w + C * np.maximum(0, 1 - y * (X @ w + b)).sum()
_ref12 = _obj(_svc.coef_[0], _svc.intercept_[0], _X12, _y12, 1.0)
def _sub_ok(fn):
    w, b = np.array([.7, -.4]), .1
    gw, gb = fn(w, b, _X12, _y12, 1.0)
    eps = 1e-6
    num_w = np.array([(_obj(w + eps*e, b, _X12, _y12, 1.) - _obj(w - eps*e, b, _X12, _y12, 1.)) / (2*eps) for e in np.eye(2)])
    num_b = (_obj(w, b + eps, _X12, _y12, 1.) - _obj(w, b - eps, _X12, _y12, 1.)) / (2*eps)
    return np.allclose(gw, num_w, atol=1e-4) and np.isclose(gb, num_b, atol=1e-4)


## Hinge loss, subgradient và huấn luyện SVM tuyến tính

Gợi ý: Chỉ các điểm có y(w·x + b) < 1 đóng góp vào subgradient. Dùng learning rate giảm dần lr0/√t và giữ lại nghiệm có mục tiêu nhỏ nhất.

In [ ]:
def svm_objective(w, b, X, y, C):
    # y ∈ {-1, +1}. Trả 0.5*||w||^2 + C * tổng hinge.
    raise NotImplementedError

def svm_subgrad(w, b, X, y, C):
    # Trả (gw, gb).
    raise NotImplementedError

def train_svm(X, y, C, steps=3000, lr0=0.01):
    # Subgradient descent từ w = 0, b = 0; trả (w, b) có mục tiêu nhỏ nhất đã gặp.
    raise NotImplementedError


In [ ]:
_report('d12-svm',
    _check('Hàm mục tiêu đúng', lambda: (np.isclose(svm_objective(np.array([1., -2.]), .5, np.array([[1., 0.], [0., 1.], [2., 2.]]), np.array([1., -1., 1.]), 2.), 2.5 + 2 * (0.0 + 0.0 + 2.5)), 'Ví dụ 3 điểm tính tay.')),
    _check('Subgradient khớp sai phân tại điểm không gấp khúc', lambda: (_sub_ok(svm_subgrad), 'Tại w = [0,7, −0,4], b = 0,1.')),
    _check('Mục tiêu sau huấn luyện ≤ 1,02 × SVC(kernel="linear")', lambda: ((lambda wb: svm_objective(wb[0], wb[1], _X12, _y12, 1.0) <= 1.02 * _ref12)(train_svm(_X12, _y12, 1.0)), f'Mục tiêu tham chiếu {_ref12:.3f}.')),
)


## Dự án

Trên Titanic (đã xử lý thiếu và one-hot), so LinearSVC và SVC(kernel="rbf") bằng cross-validation với C ∈ {0,1; 1; 10}. Ghi số vector hỗ trợ của SVC theo C.